In [3]:
!pip install -U langchain
!pip install -U langchain-cohere
!pip install -U langchain-community
!pip install -U langchain-text-splitters
!pip install -U langchain-chroma
!pip install -U pypdf
!pip install -U chromadb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.6/139.6 kB 8.5 MB/s eta 0:00:00
  Attempting uninstall: langchain
    Found existing installation: langchain 1.3.13
    Uninstalling langchain-1.3.13:
      Successfully uninstalled langchain-1.3.13
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.1/45.1 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 334.3/334.3 kB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 108.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 65.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 42.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.5 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not c

In [1]:
from google.colab import userdata
key=userdata.get('coherekey')

## Load PDF

In [5]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("/content/FadyAtia_AiEngineer_cv  - V2.pdf")

documents = loader.load()

print("Number of pages:", len(documents))

print(documents[0].page_content[:500])

ValueError: File path /content/FadyAtia_AiEngineer_cv  - V2.pdf is not a valid file or url

## Chunking

In [4]:
from langchain_cohere import CohereEmbeddings

embeddings = CohereEmbeddings(
    model="embed-v4.0",
    cohere_api_key=key
)


In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(documents)

# ChromaDB

In [8]:
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="pdf_rag"
)

## Retriever

In [9]:
retriever = vectorstore.as_retriever(
    search_kwargs={
        "k": 5
    }
)

In [10]:
query = "What is this document about?"

retrieved_docs = retriever.invoke(query)

for i, doc in enumerate(retrieved_docs):

    print(f"\n--- Document {i+1} ---")

    print(doc.page_content)

    print("Metadata:", doc.metadata)


--- Document 1 ---
causes and mitigation strategies using RAG techniques and systematic evaluation methods. 
• Explored Small Language Models (SLMs) and performed extensive benchmarking to evaluate their efficiency, reliability, and 
performance when integrated into multi-agent workflows. 
• Designed and executed research benchmarks to assess the capabilities of SLMs in agentic environments, comparing them 
against larger models in terms of accuracy, speed, and hallucination reduction. 
AI Trainee – National Telecommunication Institute (NTI Hire Ready Program) 10/2025 – 01/2026 
• Completed an intensive 4-month advanced training program focused on Machine Learning, Deep Learning, and Generative   AI 
systems.
Metadata: {'total_pages': 2, 'creator': 'Microsoft® Word LTSC', 'page': 1, 'page_label': '2', 'source': '/content/FadyAtia_AiEngineer_cv  - V2.pdf', 'producer': 'Microsoft® Word LTSC', 'moddate': '2026-06-18T16:00:06+03:00', 'creationdate': '2026-06-18T16:00:06+03:00', 'author': 

## Generation


In [12]:
from langchain_cohere import ChatCohere

llm = ChatCohere(
    model="command-a-03-2025",
    temperature=0,
    cohere_api_key=key
)


# Prompt

In [13]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant.

Answer the user's question using ONLY the context below.

If the answer cannot be found in the context,
say "I don't know."

Context:
{context}## top 1

Question:
{question}
""")

# RAG Chain

In [15]:
from langchain_core.runnables import RunnablePassthrough


In [ ]:
def format_docs(docs):
    return "\n\n".join(
        doc.page_content
        for doc in docs
    )


rag_chain = (
    {"context": retriever | format_docs, 
     "question": RunnablePassthrough()
     }
    | prompt
    | llm
)

In [17]:
question = "What is the main topic of the document?"

response = rag_chain.invoke(question)

print(response.content)

The main topic of the document is the professional profile and experience of **Fady Atia**, an **AI Engineer** specializing in **LLM-powered systems**, **Agentic workflows**, and **RAG applications**. The document highlights his education, technical skills, projects, and work experience in the field of **Artificial Intelligence** and **Machine Learning**.
